<a href="https://colab.research.google.com/github/hyeseong-shin/NLP/blob/main/Lab2_Sentiment_Analysis_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2. Sentiment Analysis
## 사전학습 모델을 활용한 영어 영화 리뷰 감성 분류

이번 실습에서는 영어 영화 리뷰를 읽고 **긍정 또는 부정으로 분류하는 모델**을 만듭니다.

예를 들어 다음 두 리뷰는 서로 다른 감성을 나타냅니다.

- “The movie was wonderful. I would watch it again.” → 긍정
- “The story was boring and the acting was terrible.” → 부정

이처럼 텍스트에 나타난 감성을 분류하는 작업을 **감성 분석(Sentiment Analysis)**이라고 합니다.

### 실습 목표

이번 실습에서는 제공된 코드를 함께 실행하며 **데이터 준비부터 모델 학습·평가·제출까지 fine-tuning의 전체 흐름**을 경험합니다.

1. 영화 리뷰 데이터의 입력과 정답 라벨을 이해합니다.
2. Tokenizer를 사용하여 텍스트를 모델의 입력으로 변환합니다.
3. 사전학습된 BERT 모델을 감성 분석 데이터로 fine-tuning합니다.
4. 검증 데이터로 모델의 성능을 평가하고 오분류 사례를 살펴봅니다.
5. 학습한 모델을 저장하여 실습 사이트에 제출합니다.

학습·평가 과정에는 Hugging Face의 `Trainer`를 사용합니다.

### 사전학습과 fine-tuning

사전학습(Pretraining)은 모델이 대량의 데이터를 통해 언어의 패턴을 학습하는 과정입니다. 이번 실습에서는 사전학습된 **BERT** 모델을 불러와 사용합니다.

BERT에 긍정·부정을 구분하는 분류기를 연결하고, 영화 리뷰와 정답 라벨로 추가 학습합니다. 이처럼 사전학습된 모델을 특정 작업에 맞게 추가 학습하는 과정을 미세조정(Fine-tuning)이라고 합니다.

이번 fine-tuning에서는 **BERT와 분류기의 가중치를 함께 업데이트**합니다.

### Baseline과 이후 개발 과제

**Baseline**은 이후 개선 실험의 결과와 비교하기 위한 기준 모델입니다. 이번 실습에서 모두 같은 데이터와 설정으로 fine-tuning한 모델을 Baseline으로 사용합니다.

실습 이후에는 **두 차례의 개발 기간** 동안 데이터나 학습 설정 등을 변경하며 모델을 개선합니다. 변경한 내용이 어떤 영향을 주었는지 Baseline과 비교합니다.

### 제출 파일

- `학번_Lab2.ipynb`: 실습 코드, 실행 결과, 질문에 대한 답변
- `submission.zip`: 학습한 모델 가중치와 설정 파일

`submission.zip`은 실습 마지막 코드 셀에서 생성합니다.

> > **중간평가와 최종평가 모두 리더보드 사이트에 모델 ZIP을 제출해야 합니다.** 중간평가는 사이트 제출로 완료되며, 최종평가는 지정된 기한에 e-Class LMS에도 Notebook(`.ipynb`)과 모델 ZIP을 제출해야 완료됩니다.

## 1. 실습 환경 준비

### GPU 설정

이번 실습에서는 PyTorch를 기반으로 BERT 모델을 학습합니다.

모델 학습에는 많은 계산이 필요합니다. 여러 계산을 병렬로 처리하는 **GPU**를 사용하면 학습 시간을 줄일 수 있습니다.

Colab 상단 메뉴에서 다음과 같이 설정하세요.

1. **런타임 → 런타임 유형 변경**
2. 하드웨어 가속기에서 **T4 GPU** 선택
3. **저장**

GPU 설정을 변경하면 런타임이 다시 연결될 수 있으므로, 라이브러리 설치와 코드 실행 전에 설정하세요.

### 라이브러리 설치

이번 실습에서는 다음 라이브러리를 사용합니다.

| 라이브러리 | 역할 |
|---|---|
| `torch` | Tensor 연산과 모델의 가중치 학습을 수행합니다. |
| `transformers` | 사전학습된 BERT와 tokenizer를 불러오고, `Trainer`로 학습·평가를 진행합니다. |
| `datasets` | 영화 리뷰 데이터를 불러오고 학습용·검증용으로 나눕니다. |
| `accelerate` | `Trainer`가 GPU 등의 실행 환경에서 동작하도록 지원합니다. |
| `scikit-learn` | Accuracy, Macro F1 등 분류 성능 지표를 계산합니다. |
| `safetensors` | 학습한 모델 가중치를 저장하는 파일 형식을 제공합니다. |

Colab에 기본으로 설치된 PyTorch를 사용하고, 나머지 라이브러리는 아래 설치 셀에서 준비합니다. 실습 코드와의 호환성을 위해 지정된 버전 조건으로 설치합니다.

**실행 순서**

1. GPU를 설정합니다.
2. 아래 라이브러리 설치 셀을 실행합니다.
3. 런타임 재시작 안내가 나타나면 재시작합니다.
4. 다음 코드 셀부터 순서대로 실행하여 GPU 연결과 라이브러리 버전을 확인합니다.

런타임을 재시작하면 이전에 만든 변수와 메모리에 불러온 모델이 초기화됩니다. 실습 도중 재시작했다면, 설치 다음의 코드 셀부터 다시 순서대로 실행하세요.

In [ ]:
# 실습에 필요한 라이브러리를 설치합니다.
# PyTorch는 Colab에 기본 설치된 버전을 사용합니다.
%pip install -q \
    "transformers==4.57.1" \
    "datasets>=3,<5" \
    "accelerate>=1,<2" \
    "scikit-learn>=1.5,<2" \
    "safetensors>=0.4.5,<1" \
    numpy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 493.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 62.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import torch
import transformers
import datasets
import accelerate
import sklearn
import safetensors
import numpy as np
import pandas as pd

# GPU 연결 여부를 확인합니다.
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU가 연결되지 않았습니다. "
        "Colab의 '런타임 → 런타임 유형 변경'에서 "
        "'T4 GPU'를 선택한 뒤 다시 실행하세요."
    )

device = torch.device("cuda")

print("사용 장치:", device)
print("GPU 이름:", torch.cuda.get_device_name(0))

# 실행 환경을 기록합니다.
print("\n라이브러리 버전")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Safetensors:", safetensors.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

사용 장치: cuda
GPU 이름: Tesla T4

라이브러리 버전
PyTorch: 2.11.0+cu130
Transformers: 4.57.1
Datasets: 4.8.5
Accelerate: 1.15.0
Scikit-learn: 1.6.1
Safetensors: 0.8.0
NumPy: 2.1.3
Pandas: 2.2.3


## 2. 감성 분석 데이터 살펴보기

이번 실습에서는 IMDb 영어 영화 리뷰 데이터를 사용합니다.

각 데이터에는 다음 두 가지 정보가 있습니다.

- `text`: 영화 리뷰
- `label`: 리뷰의 감성 정답
  - `0`: negative — 부정
  - `1`: positive — 긍정

영화 줄거리에 슬픈 사건이 등장하더라도 작성자는 영화를 긍정적으로 평가할 수 있습니다. 우리가 분류하려는 것은 줄거리 속 사건의 감정이 아니라, 작성자가 영화를 평가하는 태도입니다.

우선 IMDb 데이터셋의 공개된 train split을 불러와 실제 리뷰와 라벨을 확인합니다. 이후 이 데이터의 일부를 실습용 학습 데이터와 검증 데이터로 나눕니다.

학습 데이터(train): 모델의 가중치를 업데이트하는 데 사용합니다.

검증 데이터(validation): 가중치 업데이트에 사용하지 않고, 모델의 성능을 확인하고 개선 방법을 비교하는 데 사용합니다.

Baseline과 이후 Development 실험에서는 동일한 검증 데이터를 사용하여 결과를 비교합니다.

In [ ]:
from datasets import load_dataset
import pandas as pd
from IPython.display import display

# 공개 train split만 불러옵니다.
dataset = load_dataset("stanfordnlp/imdb", split="train")

print(dataset)
print("\n데이터 개수:", len(dataset))
print("라벨 이름:", dataset.features["label"].names)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

데이터 개수: 25000
라벨 이름: ['neg', 'pos']


In [ ]:
# negative, positive 라벨링 진행
LABEL_NAMES = {1: "positive", 0: "negative"}

# 긍정·부정 리뷰가 각각 몇 개인지 확인합니다.
label_counts = pd.Series(dataset["label"]).value_counts().sort_index()
label_counts.index = [
    LABEL_NAMES[label] for label in label_counts.index
]

display(label_counts.to_frame(name="리뷰 개수"))

# 각 클래스에서 리뷰 하나씩 확인합니다.
for label in [0, 1]:
    index = next(
        i for i, value in enumerate(dataset["label"])
        if value == label
    )

    print(f"\n정답: {LABEL_NAMES[label]}")
    print("리뷰:", dataset[index]["text"])
    print("-" * 60)

,리뷰 개수
negative,12500
positive,12500



정답: negative
리뷰: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few an

### 확인 문제

**Q1. 이 데이터에서 입력과 정답은 각각 무엇인가요?**


**Q2. 라벨 `0`과 `1`의 숫자 크기에 의미가 있나요?**

**Q3. 학습 데이터와 검증 데이터를 구분하는 이유는 무엇인가요?**

## 3. 학습 데이터와 검증 데이터 나누기

전체 리뷰 중 일부를 다음과 같이 나누어 사용합니다.

- **학습 데이터(train): 4,000개**
  - 모델의 가중치를 업데이트하는 데 사용합니다.
- **검증 데이터(validation): 1,000개**
  - 학습한 모델의 성능을 확인하고 학습 설정을 비교합니다.

두 집합에는 서로 다른 데이터 항목이 들어가도록 분리합니다.
긍정·부정 비율도 유지하도록 stratified split을 사용합니다.

또한 random seed를 고정하여, 코드를 다시 실행해도
같은 데이터 항목이 선택되도록 합니다.  
모든 Baseline·Development 실험에서 이 분할을 그대로 사용합니다.

In [ ]:
SEED = 42
TRAIN_SIZE = 4000
VAL_SIZE = 1000

# 전체 데이터에서 학습용과 검증용을 겹치지 않게 추출합니다.
split = dataset.train_test_split(
    train_size=TRAIN_SIZE,
    test_size=VAL_SIZE,
    stratify_by_column="label",
    seed=SEED,
)

train_dataset = split["train"]
val_dataset = split["test"]  # 여기서는 validation 용도로 사용합니다.

print("학습 데이터 개수:", len(train_dataset))
print("검증 데이터 개수:", len(val_dataset))

학습 데이터 개수: 4000
검증 데이터 개수: 1000


In [ ]:
def count_labels(data):
    return (
        pd.Series(data["label"])
        .value_counts()
        .reindex([0, 1], fill_value=0)
    )

distribution = pd.DataFrame({
    "train": count_labels(train_dataset),
    "validation": count_labels(val_dataset),
})

distribution.index = ["negative", "positive"]
display(distribution)

,train,validation
negative,2000,500
positive,2000,500


### 확인 문제

**Q1. stratified split을 사용하는 이유는 무엇인가요?**

학습용과 검증용 데이터에서 클래스 비율을 유지하기 위해서입니다.
이번 데이터는 긍정·부정 비율이 같으므로,
분할한 데이터에서도 두 클래스의 비율을 유지합니다.

**Q2. random seed를 고정하면 무엇이 같아지나요?**

같은 원본 데이터와 분할 코드를 사용하면
학습용·검증용으로 선택되는 데이터 항목이 같아집니다.
이 seed만으로 모델 학습 결과까지 완전히 같아지는 것은 아닙니다.

## 4. BERT Tokenizer로 입력 만들기

모델은 문자열을 그대로 계산할 수 없으므로, **tokenizer**로 텍스트를 token과 숫자 ID로 변환합니다. 이번에는 사전학습된 BERT에 대응하는 tokenizer를 사용합니다.

BERT tokenizer는 단어를 작은 조각인 **subword**로 나눌 수 있으므로, token 수와 단어 수는 다를 수 있습니다.

모델에 전달하는 입력은 다음과 같습니다.

- `input_ids`: 각 token의 숫자 ID
- `attention_mask`: 실제 입력 token은 `1`, 길이를 맞추기 위한 padding은 `0`
- `token_type_ids`: 입력 구간 구분 정보. 이번 단일 리뷰 입력에서는 모두 `0`

BERT는 입력 시작에 `[CLS]`, 끝에 `[SEP]` 특수 token을 추가합니다.

이번 Baseline은 **특수 token을 포함해 최대 128 token**을 사용하며, 초과하는 뒷부분은 잘라냅니다(truncation). 따라서 긴 리뷰는 전체 내용이 모델에 전달되지 않을 수 있습니다.

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "google-bert/bert-base-uncased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=False,
)

print("Tokenizer:", type(tokenizer).__name__)
print("어휘 크기:", tokenizer.vocab_size)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer: BertTokenizerFast
어휘 크기: 30522


In [ ]:
# 예시 문장 입력해보기
example_text = "The movie was surprisingly enjoyable! "

encoded = tokenizer(
    example_text,
    truncation=True,
    max_length=MAX_LENGTH,
)

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])

print("원문:", example_text)
print("Tokens:", tokens)
print("Input IDs:", encoded["input_ids"])
print("Attention mask:", encoded["attention_mask"])
print("Token type IDs:", encoded["token_type_ids"])

원문: The movie was surprisingly enjoyable! 
Tokens: ['[CLS]', 'the', 'movie', 'was', 'surprisingly', 'enjoyable', '!', '[SEP]']
Input IDs: [101, 1996, 3185, 2001, 10889, 22249, 999, 102]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1]
Token type IDs: [0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

# 여러 리뷰를 묶어서 토큰화합니다.
# padding은 이후 학습 batch를 구성할 때 추가합니다.
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

print(tokenized_train)
print("\n첫 리뷰의 token 수:", len(tokenized_train[0]["input_ids"]))
print("첫 리뷰의 정답:", tokenized_train[0]["label"])

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4000
})

첫 리뷰의 token 수: 128
첫 리뷰의 정답: 1


## 5. 사전학습 모델 불러오기

사전학습된 BERT에 긍정·부정을 분류하는 **분류기(classification head)** 를
연결하여 사용합니다.

- **BERT:** 입력 리뷰를 숫자 표현으로 변환합니다.
- **분류기:** 이 표현을 받아 부정·긍정 클래스의 점수를 계산합니다.

BERT는 사전학습된 가중치를 불러오지만,
이번 감성 분류용 분류기는 새로 초기화됩니다.

따라서 아직 학습하지 않은 모델의 감성 예측은 신뢰할 수 없습니다.
이후 fine-tuning에서는 BERT와 분류기 가중치를 함께 업데이트합니다.

라벨 순서는 다음과 같이 고정합니다.

- `0`: negative
- `1`: positive

In [ ]:
from transformers import AutoModelForSequenceClassification, set_seed

# 분류기의 초기값 등 학습에 사용되는 seed를 설정합니다.
set_seed(SEED)

ID2LABEL = {0: "negative", 1: "positive"}
LABEL2ID = {"negative": 0, "positive": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    trust_remote_code=False,
    use_safetensors=True,
)

model = model.to(device)

print("모델 클래스:", type(model).__name__)
print("분류할 클래스 수:", model.config.num_labels)
print("라벨 설정:", model.config.id2label)
print("모델이 올라간 장치:", next(model.parameters()).device)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


모델 클래스: BertForSequenceClassification
분류할 클래스 수: 2
라벨 설정: {0: 'negative', 1: 'positive'}
모델이 올라간 장치: cuda:0


In [ ]:
sample_inputs = tokenizer(
    example_text,
    return_tensors="pt",  # 리스트 대신 PyTorch tensor로 반환
    truncation=True,
    max_length=MAX_LENGTH,
)

# 모델과 입력을 같은 장치에 둡니다.
sample_inputs = {
    key: value.to(device)
    for key, value in sample_inputs.items()
}

model.eval()

with torch.inference_mode():
    outputs = model(**sample_inputs)

logits = outputs.logits
predicted_id = logits.argmax(dim=-1).item()

print("입력:", example_text)
print("Logits shape:", tuple(logits.shape))
print("Logits:", logits.cpu().tolist())
print("학습 전 예측:", ID2LABEL[predicted_id])

입력: The movie was surprisingly enjoyable! 
Logits shape: (1, 2)
Logits: [[0.06148669868707657, 0.0955883264541626]]
학습 전 예측: positive


### 확인 문제

**Q1. Logits shape `(1, 2)`는 무엇을 의미하나요?**

**Q2. Logits는 확률인가요?**


**Q3. 학습 전 예측이 맞았다면 이미 감성 분류를 잘한다고 볼 수 있나요?**

## 6. Baseline 모델 학습

Baseline은 이후 실험과 비교할 기준 모델입니다.
먼저 모두 같은 설정으로 학습합니다.

- **Epoch: 1**
  - 학습 데이터 전체를 한 번 사용합니다.
- **Batch size: 16**
  - 리뷰 16개를 묶어 한 번의 가중치 업데이트에 사용합니다.
- **Learning rate: 0.00002**
  - 가중치를 업데이트하는 크기를 조절합니다.

학습에는 Hugging Face의 `Trainer`를 사용합니다.
Trainer가 예측, 손실 계산, 가중치 업데이트를 반복합니다.

길이가 다른 리뷰를 하나의 batch로 묶을 때는
해당 batch의 가장 긴 입력에 맞춰 padding을 추가합니다.

검증 지표는 다음과 같습니다.

- **Accuracy:** 전체 리뷰 중 올바르게 분류한 비율
- **Macro F1:** 각 클래스의 F1을 구한 뒤 평균한 값

두 지표 모두 이번 실습에서는 0~1 범위이며, 높을수록 좋습니다.
검증 데이터는 가중치 업데이트에 사용하지 않습니다.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels,
            predictions,
            labels=[0, 1],
            average="macro",
            zero_division=0,
        ),
    }

# 각 batch의 가장 긴 입력에 맞춰 padding을 추가합니다.
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt",
)

In [ ]:
baseline_args = TrainingArguments(
    output_dir="./lab2_baseline",

    # 학습 설정
    learning_rate=2e-5,
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,

    # 매 epoch 종료 시 validation 평가
    eval_strategy="epoch",

    # 제출 파일은 이후 별도 셀에서 저장
    save_strategy="no",

    # 학습 진행 기록
    logging_strategy="steps",
    logging_steps=50,
    report_to="none",

    # 난수 설정
    seed=SEED,
    data_seed=SEED,

    # 처음에는 혼합 정밀도를 사용하지 않습니다.
    fp16=False,
    bf16=False,
)

baseline_trainer = Trainer(
    model=model,
    args=baseline_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
baseline_train_result = baseline_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.363700,0.352942,0.838000,0.837490


### 확인 문제

**Q1. 데이터 4,000개, batch size 16, epoch 1이면
가중치 업데이트는 몇 번 이루어지나요?**


**Q2. 학습 중 validation 데이터도 가중치 업데이트에 사용되나요?**

**Q3. Training loss가 낮아지면 validation 성능도 반드시 좋아지나요?**


## 7. Baseline 평가 및 오분류 분석

학습한 모델을 validation 데이터로 평가합니다.
점수와 오분류 사례는 실행 결과에서 확인하세요.
실행 환경에 따라 결과가 달라질 수 있습니다.

### 평가 지표 읽기

- **Accuracy:** 전체 리뷰 중 올바르게 분류한 비율
- **Precision:** 특정 클래스로 예측한 리뷰 중 실제로 그 클래스인 비율
- **Recall:** 특정 클래스의 리뷰 중 모델이 올바르게 찾아낸 비율
- **F1:** Precision과 Recall을 함께 반영한 지표
- **Macro F1:** 각 클래스의 F1을 단순 평균한 값
- **Support:** 각 클래스의 실제 리뷰 개수



In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# 검증 데이터 전체에 대한 예측
baseline_prediction = baseline_trainer.predict(tokenized_val)

val_logits = baseline_prediction.predictions
val_predictions = np.argmax(val_logits, axis=-1)
val_labels = baseline_prediction.label_ids

baseline_metrics = {
    "accuracy": accuracy_score(val_labels, val_predictions),
    "macro_f1": f1_score(
        val_labels,
        val_predictions,
        labels=[0, 1],
        average="macro",
        zero_division=0,
    ),
}

print("Baseline:", baseline_metrics)

# 행: 정답 / 열: 예측
cm = confusion_matrix(
    val_labels,
    val_predictions,
    labels=[0, 1],
)

display(pd.DataFrame(
    cm,
    index=["정답 negative", "정답 positive"],
    columns=["예측 negative", "예측 positive"],
))

print(classification_report(
    val_labels,
    val_predictions,
    labels=[0, 1],
    target_names=["negative", "positive"],
    zero_division=0,
))

Baseline: {'accuracy': 0.838, 'macro_f1': 0.8374903697996918}


,예측 negative,예측 positive
정답 negative,391,109
정답 positive,53,447


              precision    recall  f1-score   support

    negative       0.88      0.78      0.83       500
    positive       0.80      0.89      0.85       500

    accuracy                           0.84      1000
   macro avg       0.84      0.84      0.84      1000
weighted avg       0.84      0.84      0.84      1000



### Confusion Matrix 읽기

행은 실제 정답, 열은 모델의 예측입니다.

- 대각선: 올바르게 분류한 리뷰 수
- 나머지 칸: 잘못 분류한 리뷰 수

정답이 negative인데 positive로 예측한 경우와,
정답이 positive인데 negative로 예측한 경우를 비교하세요.

In [ ]:
error_indices = np.flatnonzero(val_predictions != val_labels)

print("오분류 개수:", len(error_indices))

for index in error_indices[:3]:
    index = int(index)

    print(f"\n검증 데이터 index: {index}")
    print("정답:", ID2LABEL[int(val_labels[index])])
    print("예측:", ID2LABEL[int(val_predictions[index])])

    # 모델에 실제로 들어간 부분을 복원해서 확인합니다.
    print("\n모델 입력 부분:")
    print(tokenizer.decode(
        tokenized_val[index]["input_ids"],
        skip_special_tokens=True,
    ))

    print("\n전체 리뷰:")
    print(val_dataset[index]["text"])
    print("=" * 70)

오분류 개수: 162

검증 데이터 index: 4
정답: positive
예측: negative

모델 입력 부분:
this musical is decidedly mixed, and none of the elements really fit together, but it somehow manages to be mostly enjoyable. the plot contains some of the elements of wodehouse ' s novel, but none of its virtues, though he co - wrote the script. the songs, though charming, have nothing to do with this particular film, and are unusually crudely squeezed into the plot, even by pre - oklahoma standards. burns and allen do their usual shtick quite competently, but it misses the tone of the rest of the film by about forty iq points. < br / > < br / > there are a few

전체 리뷰:
This musical is decidedly mixed, and none of the elements really fit together, but it somehow manages to be mostly enjoyable. The plot contains some of the elements of Wodehouse's novel, but none of its virtues, though he co-wrote the script. The songs, though charming, have nothing to do with this particular film, and are unusually crudely squeezed into 



### 오분류 분석

모델에 실제로 입력된 부분과 전체 리뷰를 함께 읽으세요.

다음은 오분류를 설명할 수 있는 가설입니다.

- 긍정·부정 평가가 한 리뷰에 함께 나타남
- 반어 또는 간접적인 평가 표현이 등장함
- 줄거리의 부정적인 사건과 작성자의 부정적인 평가를 혼동함
- 길이 제한으로 중요한 평가 부분이 잘림

점수를 읽는 것만으로 모델의 내부 판단 원인을 확정할 수는 없습니다.
리뷰에서 찾은 근거를 바탕으로 가능한 원인을 설명하세요.

### 확인 문제와 모범 답안

**Q1. 오분류 리뷰 두 개를 선택하여 가능한 원인을 설명하세요.**

**Q2. 영화 줄거리에 슬픈 사건이 나오면 부정 리뷰인가요?**


**Q3. 입력 길이를 늘리면 모든 오분류가 해결될까요?**


## 8. 제출용 모델 ZIP 생성

학습한 모델을 자동평가용 ZIP 파일로 저장합니다. 아래 코드는 Baseline 모델을 저장하는 예시입니다. Development에서는 자유롭게 학습 코드를 작성하되, 제출 규격을 지켜야 합니다.

### ZIP 파일 구성

ZIP 최상위에 다음 세 파일을 넣습니다. 상위 폴더 없이 파일이 바로 들어가야 합니다.

- `manifest.json`: 태스크, 모델, 라벨 등의 제출 정보
- `config.json`: 모델 구조와 라벨 설정
- `model.safetensors`: 제출할 모델의 학습된 가중치

`config.json`과 `model.safetensors`는 제출할 모델의 `save_pretrained()`로 생성합니다. 가중치는 `safe_serialization=True`로 저장하세요.

지정된 모델과 라벨 순서(`0`: negative, `1`: positive)를 유지해야 합니다. Tokenizer는 서버에서 제공하므로 포함하지 않습니다. 학습 코드, 데이터, optimizer 상태 등 다른 파일도 ZIP에 넣지 않습니다.

**Development 모델을 제출할 때는 아래 저장 코드가 실제로 개선한 모델을 가리키는지 확인하세요.** ZIP 생성 셀을 다시 실행하면 기존 `submission.zip`을 덮어씁니다.

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

export_dir = Path("./lab2_baseline_export")
zip_path = Path("./submission.zip")

# 이 셀에서 사용하는 내보내기 폴더만 정리합니다.
if export_dir.exists():
    shutil.rmtree(export_dir)

export_dir.mkdir(parents=True)
zip_path.unlink(missing_ok=True)

# 학습한 Baseline 모델을 safetensors 형식으로 저장합니다.
baseline_trainer.model.save_pretrained(
    export_dir,
    safe_serialization=True,
    max_shard_size="2GB",
)

manifest = {
    "format_version": "1.0",
    "task": "sentiment",
    "base_model": BASE_MODEL,
    "architecture": "BertForSequenceClassification",
    "artifact_type": "full_model",
    "num_labels": 2,
    "label_mapping": {
        str(label_id): label_name
        for label_id, label_name in ID2LABEL.items()
    },
}

with (export_dir / "manifest.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

# 필수 파일만 생성되었는지 확인합니다.
allowed_files = {
    "manifest.json",
    "config.json",
    "model.safetensors",
}

entries = list(export_dir.iterdir())
generated_files = {p.name for p in entries}

if any(p.is_symlink() or not p.is_file() for p in entries):
    raise ValueError("제출 폴더에 일반 파일이 아닌 항목이 있습니다.")

if generated_files != allowed_files:
    raise ValueError(
        f"제출 파일 구성이 다릅니다.\n"
        f"누락: {allowed_files - generated_files}\n"
        f"추가: {generated_files - allowed_files}"
    )

# ZIP 안에 상위 폴더 없이 세 파일을 바로 넣습니다.
with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in sorted(allowed_files):
        archive.write(
            export_dir / filename,
            arcname=filename,
        )

# ZIP 파일 목록과 압축 무결성을 확인합니다.
with zipfile.ZipFile(zip_path) as archive:
    assert len(archive.namelist()) == 3
    assert set(archive.namelist()) == allowed_files
    assert archive.testzip() is None
    print("ZIP 내부 파일:", archive.namelist())

print(f"\n생성 완료: {zip_path.name}")
print(f"파일 크기: {zip_path.stat().st_size / 1024**2:.1f} MiB")
print("내보낸 모델: Baseline")

ZIP 내부 파일: ['config.json', 'manifest.json', 'model.safetensors']

생성 완료: submission.zip
파일 크기: 386.8 MiB
내보낸 모델: Baseline


In [ ]:
from google.colab import files

files.download(str(zip_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 사이트 제출
1. 생성한 `submission.zip`을 다운로드합니다.
2. 사이트에서 제출 유형을 **Baseline**으로 선택합니다.
3. ZIP 파일을 함께 업로드합니다.

학번은 수강생 확인과 제출 기록 연결에만 사용되며,
공개 리더보드에는 표시되지 않습니다.

- 사이트는 점수 확인용입니다.

## 9. Development 과제 안내

이번 과제에서는 실습에서 만든 Baseline을 바탕으로
영어 영화 리뷰 감성 분류 모델을 개선합니다.

중간 제출에서는 첫 번째 개선 실험을 진행하고,
최종 제출에서는 그 결과를 바탕으로 추가 개선합니다.
각 제출 전에는 여러 실험을 진행할 수 있습니다.

### 1. 개선 방향

| 개선 방향 | 실험 예시 |
|---|---|
| 학습 데이터 추가 | 추가적인 영어 영화 리뷰 감성 데이터를 찾아 학습 |
| 데이터 정제 | 중복 리뷰 제거, 빈 리뷰 및 라벨 오류가 의심되는 항목 확인 |
| 텍스트 전처리 | HTML 태그나 불필요한 공백 정리 |
| 학습 설정 변경 | learning rate, epoch, batch size, weight decay 변경 |

표의 내용은 예시입니다. 실험 규칙 안에서 다른 방법도 자유롭게 시도하세요.

데이터를 추가하거나 전처리한다고 성능이 반드시 좋아지는 것은 아닙니다.
변경 전후 결과를 비교하여 실제 효과를 확인하세요.

### 2. 실험 규칙

- 지정된 BERT 모델과 tokenizer를 사용합니다.
- 긍정·부정 라벨 순서를 유지합니다: `0`은 negative, `1`은 positive입니다.
- 제공된 validation은 학습에 사용하지 않고, 실험 간 성능 비교에 동일하게 사용합니다.
- 추가 데이터는 출처·사용한 split·개수를 기록하고, 평가용 데이터나 제공된 validation과 중복되는 리뷰는 학습에서 제외합니다.

### 3. 중간 제출

Baseline을 바탕으로 하나 이상의 개선 실험을 진행합니다.
실험 코드와 실행 결과가 포함된 모델 ZIP을 제출합니다.

중간 제출에서는 별도 보고서나 개발 요약을 제출하지 않습니다.

### 4. 평가 데이터 구분

- **공개 validation:** 직접 성능을 비교하고 오류를 분석하는 용도
- **리더보드 비공개 평가셋:** 사이트에서 개발 중 성능을 확인하는 용도
- **최종 비공개 test:** 최종 제출 모델의 공식 평가 용도

평가 데이터가 다르므로 validation 점수와 사이트 점수는 다를 수 있습니다.
리더보드 점수만으로 최종 평가 결과가 보장되지는 않습니다.

### 5. 제출 방법

사이트에서는 제출 유형을 **Development**로 선택하고,
다음 파일을 업로드합니다.

- 개선한 모델로 생성한 `submission.zip`

ZIP을 만들 때 실제로 제출하려는 개선 모델을 저장했는지 확인하세요.
Baseline 모델을 다시 저장하지 않도록 주의합니다.
